In [1]:
#! pip install datasets
# ! pip install ragas

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [19]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

In [16]:
#data loading
# loader = TextLoader(r"C:\Users\ekrsinh\OneDrive - Ericsson\GFGO\AgenticAI_2026\DocumentQA_advance\explanation.txt")
loader = TextLoader("explanation.txt",encoding="utf-8")
document = loader.load()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500,chunk_overlap = 50)
chunks = text_splitter.split_documents(document)

In [21]:
#embedding
embeddings = OpenAIEmbeddings()

In [ ]:
from langchain_community.vectorstores import FAISS
vector_db = FAISS.from_documents(chunks,embedding=embeddings)

In [25]:
retriver = vector_db.as_retriever()

In [26]:
# LLM
llm = ChatOpenAI(temperature=0.2)

In [27]:
#Define prompt template
template = """ You are an asisstant for question answering tasks.
Use the following peice of conrexts to answer the question.
If you dont know the answer,just say tyou do not know the answer.
Use tow sentances maximum and keep the answer consise.
Question : {question}
Context : {context}
Answer :
"""




In [28]:
# Create chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser



In [29]:
prompt = ChatPromptTemplate.from_template(template)

In [30]:
#setting up the RAG pipeline
chain = (
    {"context":retriver,"question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)



In [31]:
chain.invoke("what is the this document about?")

'The document is about the design and functional requirements for an AI-powered assistant that interacts with SAP transactional data through a conversational interface, primarily targeting users in finance and accounts payable. It aims to simplify complex queries by providing natural language access to various SAP data sources while meeting enterprise-grade requirements.'

In [33]:
questions = [
    "What is the main objective of building the AI-powered assistant for SAP data integration?",
    "How does the assistant handle natural language queries and multi-turn conversations?",
    "What types of SAP data sources are integrated into the system, and why is this integration important?",
    "What kind of information can the assistant provide related to invoices, vendors, and purchase orders?",
    "What are the key non-functional requirements of the system, and how do they ensure performance, security, and scalability?"
]


In [34]:
ground_truth = [
    "The main objective is to create a unified AI assistant that consolidates data from multiple SAP transactions and allows users to query complex operational and financial data using natural language.",
    "The assistant uses natural language understanding (NLU) to interpret user queries, map them to SAP data fields, and supports multi-turn conversations by maintaining context for follow-up queries.",
    "The system integrates data from SAP transactions such as invoice (FB03, FBL1N, FBL3N), vendor (FK03, BP), purchase order (ME23N, ME2N, ME2L), and cost/project data (CJ03, CN23, KS03) to enable real-time, unified access to enterprise data.",
    "It provides details like invoice status, due dates, approval delays, vendor block status, balances, PO status, requisitioner, payment terms, and delivery or invoicing information.",
    "Key non-functional requirements include high performance (fast response times), security (role-based access and data masking), scalability (support for 300+ users), auditability (logging of queries), and a modular architecture for extensibility."
]

In [49]:
answer = []
content = []

In [50]:
for query in questions:
    answer.append(chain.invoke(query))
    # content.append([docs.page_content for docs in retriver._get_relevant_documents(query)]) #Fetching retrieved documents
    content.append([doc.page_content for doc in retriver.invoke(query)]) #Fetching retrieved documents

In [58]:
# lets prepare one dict to fed into Ragas as input
data = {
    "user_input" : questions,
    "reference" : ground_truth,
    "response" : answer,
    "retrieved_contexts" : content
}

In [59]:
from datasets import Dataset
dataset = Dataset.from_dict(data)



In [60]:
print(dataset.features)

{'user_input': Value('string'), 'reference': Value('string'), 'response': Value('string'), 'retrieved_contexts': List(Value('string'))}


In [61]:
dataset

Dataset({
    features: ['user_input', 'reference', 'response', 'retrieved_contexts'],
    num_rows: 5
})

In [62]:
# evaluation by Ragas
from ragas import evaluate
from ragas.metrics import (faithfulness,answer_relevancy,context_recall,context_precision)

C:\Users\ekrsinh\AppData\Local\Temp\ipykernel_28912\719768838.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (faithfulness,answer_relevancy,context_recall,context_precision)
C:\Users\ekrsinh\AppData\Local\Temp\ipykernel_28912\719768838.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (faithfulness,answer_relevancy,context_recall,context_precision)
C:\Users\ekrsinh\AppData\Local\Temp\ipykernel_28912\719768838.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from r

In [63]:
result = evaluate(dataset= dataset,metrics=[faithfulness,answer_relevancy,context_recall,context_precision],llm=llm,embeddings=embeddings )

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
